In [ ]:

# 1. Import libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# ---------------------------------------
# STEP 1: LOAD AND PREPROCESS DATA
# ---------------------------------------

# Load dataset
train_data = pd.read_csv("sign_mnist_train.csv")
test_data = pd.read_csv("sign_mnist_test.csv")

# Separate images and labels
x_train = train_data.drop("label", axis=1).values
y_train = train_data["label"].values

x_test = test_data.drop("label", axis=1).values
y_test = test_data["label"].values

# Normalize pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape images
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# Select Y (label 24) as anomaly
anomaly_label = 24

# Keep only 23 normal classes for training
x_normal = x_train[y_train != anomaly_label]

print("Normal training images:", len(x_normal))
print("Anomaly class: Y (label 24)")


# ---------------------------------------
# STEP 2: BUILD AUTOENCODER MODEL
# ---------------------------------------

model = tf.keras.Sequential([

    # Encoder
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),

    # Decoder
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(784, activation="sigmoid"),
    tf.keras.layers.Reshape((28, 28, 1))

])

# Compile model
model.compile(
    optimizer="adam",
    loss="mse"
)


# ---------------------------------------
# STEP 3: TRAIN AUTOENCODER
# ---------------------------------------

history = model.fit(
    x_normal,
    x_normal,
    epochs=10,
    batch_size=128,
    validation_split=0.1
)


# ---------------------------------------
# STEP 4: CALCULATE RECONSTRUCTION ERROR
# ---------------------------------------

# Reconstruct test images
reconstructed = model.predict(x_test)

# Calculate reconstruction error
errors = np.mean(
    np.square(x_test - reconstructed),
    axis=(1, 2, 3)
)

# Calculate threshold using normal test images
normal_errors = errors[y_test != anomaly_label]

threshold = np.percentile(normal_errors, 95)

print("Threshold:", threshold)


# ---------------------------------------
# STEP 5: DETECT ANOMALIES
# ---------------------------------------

# Predict anomaly
predictions = (errors > threshold).astype(int)

# Actual labels: 1 = anomaly, 0 = normal
actual = (y_test == anomaly_label).astype(int)

# Calculate accuracy
accuracy = np.mean(predictions == actual)

print("Detection Accuracy:", accuracy)

# Count detected anomalies
print("Actual anomalies:", np.sum(actual))
print("Detected anomalies:", np.sum(predictions))


# ---------------------------------------
# STEP 6: DISPLAY RESULTS
# ---------------------------------------

# Display original and reconstructed image
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(x_test[0].reshape(28, 28), cmap="gray")
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(reconstructed[0].reshape(28, 28), cmap="gray")
plt.title("Reconstructed Image")
plt.axis("off")

plt.show()

# Plot training loss
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Autoencoder Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'ecg.csv'

## Practical 3F – Sign Language Anomaly Detection using Autoencoder

Problem statement: Using the Sign Language MNIST dataset, select one of the 24 alphabet classes (label Y–24) as an anomaly. Train an Autoencoder using the remaining 23 classes and detect the selected class using reconstruction error.

### 1. Understand the practical

In this practical, we use images of hand gestures representing sign language alphabets.

Our goal is to teach the model 23 alphabet classes and keep one alphabet (Y) as an unknown class.

- 23 classes → Normal data.
- Y (label 24) → Anomaly.
- If the reconstruction error is high, the image is classified as an anomaly.

[Finger Spelling the Alphabet in American Sign Language. Letter Y](https://images.openai.com/static-rsc-4/mTG2_7hYIHXCjhKU_xNK1cVNjrjil__C5a9tmfhbAq4XEcgA5oR-pGeNAP2o9nJXB4R_W4kAbVqOfDzVFFX4rG7TJfNWLtj1_FF3UfUXRZN87DueHjCyqn5xJ-LvZcizp3-ZgBXo57ub7kZlF3ocFREEYhbrrED800u_0pG90lpDVBbGl4Xjz7E7haq87aY6?purpose=inline)

### 2. Understand the working

Sign Language MNIST Dataset

28 × 28 hand gesture images

Select 23 Normal Classes

Keep Y (label 24) separate

Autoencoder Training

Encoder → Decoder

Reconstruction Error

Anomaly Detection

High error = Y alphabet

### 3. Explanation of your code

Step 1 – Import libraries

```
import pandas as pdimport numpy as npimport tensorflow as tfimport matplotlib.pyplot as plt
```

- Pandas: loads CSV files.
- NumPy: mathematical operations.
- TensorFlow: builds the Autoencoder.
- Matplotlib: displays images and graphs.

Step 2 – Load dataset

```
train_data = pd.read_csv("sign_mnist_train.csv")test_data = pd.read_csv("sign_mnist_test.csv")
```

Loads training and testing images from your local CSV files.

Step 3 – Separate images and labels

```
x_train = train_data.drop("label", axis=1).valuesy_train = train_data["label"].values
```

- `x_train` contains image pixels.
- `y_train` contains alphabet labels.

Each image has 784 pixels (28 × 28).

Step 4 – Normalize images

```
x_train = x_train / 255.0x_test = x_test / 255.0
```

Pixel values originally range from 0 to 255.

Dividing by 255 converts them into 0 to 1.

Why? It helps the neural network learn more effectively.

Step 5 – Reshape images

```
x_train = x_train.reshape(-1, 28, 28, 1)x_test = x_test.reshape(-1, 28, 28, 1)
```

Converts each image from 784 pixel values into a 28 × 28 grayscale image.

The `1` represents one grayscale channel.

Step 6 – Select Y as anomaly

```
anomaly_label = 24x_normal = x_train[y_train != anomaly_label]
```

This is one of the most important steps.

- `y_train != 24` selects all images except Y.
- The remaining 23 classes are used for training.
- Y is excluded from training.

Step 7 – Build Autoencoder

```
model = tf.keras.Sequential([    tf.keras.layers.Input(shape=(28, 28, 1)),    tf.keras.layers.Flatten(),    tf.keras.layers.Dense(128, activation="relu"),    tf.keras.layers.Dense(64, activation="relu"),    tf.keras.layers.Dense(128, activation="relu"),    tf.keras.layers.Dense(784, activation="sigmoid"),    tf.keras.layers.Reshape((28, 28, 1))])
```

Understand the layers:

| Layer      | Purpose                          |
| ---------- | -------------------------------- |
| Input      | Receives 28 × 28 image           |
| Flatten    | Converts image into 784 values   |
| Dense(128) | Learns image features            |
| Dense(64)  | Compresses information           |
| Dense(128) | Starts reconstruction            |
| Dense(784) | Reconstructs pixel values        |
| Reshape    | Converts values back to an image |

Remember:

- Encoder = Flatten → Dense 128 → Dense 64.
- Decoder = Dense 128 → Dense 784 → Reshape.

Step 8 – Compile model

```
model.compile(    optimizer="adam",    loss="mse")
```

- Adam: updates model weights.
- MSE: calculates the difference between original and reconstructed images.

Step 9 – Train Autoencoder

```
history = model.fit(    x_normal,    x_normal,    epochs=10,    batch_size=128,    validation_split=0.1)
```

Why do we provide `x_normal` twice?

Because an Autoencoder learns to reconstruct its own input.

- Epochs = 10.
- Batch size = 128.
- Validation split = 10%.

Step 10 – Reconstruct test images

```
reconstructed = model.predict(x_test)
```

The trained model reconstructs every test image.

Step 11 – Calculate reconstruction error

```
errors = np.mean(    np.square(x_test - reconstructed),    axis=(1, 2, 3))
```

Compares original images with reconstructed images.

- Low error → model recognizes a familiar pattern.
- High error → model finds the image unusual.

Step 12 – Set threshold

```
normal_errors = errors[y_test != anomaly_label]threshold = np.percentile(normal_errors, 95)
```

Calculates the 95th percentile using normal test images.

This becomes the boundary for identifying anomalies.

Step 13 – Detect anomalies

```
predictions = (errors > threshold).astype(int)actual = (y_test == anomaly_label).astype(int)
```

Here:

| Value | Meaning         |
| ----- | --------------- |
| 0     | Normal alphabet |
| 1     | Anomaly (Y)     |

If reconstruction error exceeds the threshold, the model predicts an anomaly.

Step 14 – Display results

```
plt.imshow(x_test[0].reshape(28, 28), cmap="gray")
```

Displays the original image.

Your second subplot displays the reconstructed image.

The training-loss graph shows how the model's reconstruction loss changes across epochs.

### 4. Important viva questions

Q1. What is an Autoencoder?

A neural network that learns to compress and reconstruct input data.

Q2. What is an anomaly in this practical?

The selected Y alphabet class (label 24).

Q3. Why do we train on only 23 classes?

So the model learns the normal alphabet patterns without learning the selected anomaly class.

Q4. What is Flatten?

It converts a 2D image into a one-dimensional array.

Q5. What is Reshape?

It converts the reconstructed 784 pixel values back into a 28 × 28 image.

Q6. What is reconstruction error?

The difference between the original image and reconstructed image.

Q7. Why do we use MSE?

To measure the squared difference between original and reconstructed pixel values.

Q8. What is the purpose of the threshold?

It decides whether an image is normal or anomalous.

Q9. What is the difference between Encoder and Decoder?

Encoder compresses information, while Decoder reconstructs the original input.

Q10. Why do we use Sigmoid in the output layer?

Because normalized pixel values are between 0 and 1, and Sigmoid produces values in that range.

### 5. What to tell the examiner

"I implemented anomaly detection using an Autoencoder on the Sign Language MNIST dataset. First, I loaded the local training and testing datasets, normalized the pixel values and reshaped the images into 28 by 28 dimensions. I selected Y, label 24, as the anomaly and trained the Autoencoder using the remaining 23 classes. The model learns to reconstruct normal images. Finally, I calculated reconstruction errors, set a 95th percentile threshold using normal test images, and identified images with higher errors as anomalies."

### 6. Quick revision

| Parameter      | Value               |
| -------------- | ------------------- |
| Dataset        | Sign Language MNIST |
| Image size     | 28 × 28             |
| Total classes  | 24                  |
| Normal classes | 23                  |
| Anomaly        | Y (label 24)        |
| Encoder        | 128 → 64            |
| Decoder        | 128 → 784           |
| Optimizer      | Adam                |
| Loss           | MSE                 |
| Epochs         | 10                  |
| Batch size     | 128                 |
| Threshold      | 95th percentile     |

One important correction

Your code calculates the threshold using normal images from the test set:

```
normal_errors = errors[y_test != anomaly_label]
```

For a proper evaluation, use a separate normal validation set to select the threshold, rather than using test data. Also, the reported accuracy alone may not tell the full story; precision, recall and F1-score are useful for anomaly detection.

Remember the main concept: Train on 23 classes, keep Y outside training, calculate reconstruction error, and detect Y using the threshold.